# FNO Darcy-flow surrogate — 2× T4 scaling benchmark on Kaggle

**Before running:** open the right-hand **Settings** panel and set
- **Accelerator → GPU T4 ×2**
- **Internet → On** (needed to `git clone` and `pip install`)

Then **Run All**. This clones the repo, generates the Darcy dataset, and runs the
1-GPU vs 2-GPU scaling benchmark, printing the table you paste into the README.

In [ ]:
# 0. Confirm both GPUs are visible
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 1. Get the code.
REPO_URL = 'https://github.com/AbdullahRasheed45/fno-darcy-flow.git'
import os
os.chdir('/kaggle/working')
!rm -rf fno-darcy-flow
!git clone $REPO_URL
os.chdir('/kaggle/working/fno-darcy-flow')
!pwd && ls

In [ ]:
# 2. Dependencies. Kaggle's base image already ships a CUDA build of torch,
#    so we only add scipy/matplotlib if missing (torch stays untouched).
!pip install -q scipy matplotlib
import torch; print('torch', torch.__version__, 'CUDA', torch.cuda.is_available(), 'GPUs', torch.cuda.device_count())

In [ ]:
# 3. Generate the dataset (~1200 solved PDEs at 64x64; a few minutes on CPU)
!python -m src.data.darcy --n-samples 1200 --grid 64 --seed 0 --out data/darcy_train.npz

In [ ]:
# 4. THE SCALING BENCHMARK: same training on 1 GPU then 2 GPUs (DDP), with AMP.
#    Prints a Markdown table -> paste it into the README scaling section.
!python -m src.benchmark --data data/darcy_train.npz --gpus 1,2 --epochs 100 \
    --batch-size 16 --modes 12 --width 32 --amp --out scaling.md

In [ ]:
# 5. Show the table
from IPython.display import Markdown, display
display(Markdown(open('scaling.md').read()))

In [ ]:
# 6. Train one good model and render prediction vs. ground truth for the README
!torchrun --standalone --nproc_per_node=2 -m src.train \
    --data data/darcy_train.npz --epochs 150 --batch-size 16 --amp \
    --out checkpoints/fno_darcy.pt
!python -m src.infer --checkpoint checkpoints/fno_darcy.pt \
    --data data/darcy_train.npz --n-show 3 --out docs/prediction.png

In [ ]:
# 7. Display the figure
from IPython.display import Image
Image('docs/prediction.png')

## Save your results back out
`scaling.md`, `checkpoints/fno_darcy.pt`, and `docs/prediction.png` are under
`/kaggle/working` — they appear in the notebook's **Output** tab after you
commit (Save Version). Download them, commit the table + figure to your repo,
and you have real, defensible numbers.